# InventoryPulse — Microsoft Agent Framework on Fabric
Install the project wheel plus its `agentic` dependencies in a Fabric Environment and attach a Lakehouse. See `docs/agent-framework-handoff.md`. This notebook does not create tenant identities or model deployments.

In [ ]:
from pathlib import Path
from datetime import date
import json, os
from uuid import uuid4
from inventorypulse.agentic import Runtime, run_workflow
from inventorypulse.azure_clients import AzureClients

# Configure these in the approved notebook environment; do not commit credentials.
# Requires an attached Lakehouse, published Data Agent, model deployment and Entra credentials.
AS_OF = "2026-10-31"
WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
DATA_AGENT_ID = os.environ["FABRIC_DATA_AGENT_ID"]
ROOT = Path("/lakehouse/default/Files/inventorypulse")
DESTINATION = ROOT / "retrieval" / uuid4().hex

async with AzureClients() as clients:
    runtime = Runtime(
        data=ROOT / "input", cutoff=date.fromisoformat(AS_OF),
        fabric_workspace=WORKSPACE_ID, fabric_data_agent=DATA_AGENT_ID,
        fabric_destination=DESTINATION,
        fabric_credential=clients.credentials["retrieval"], identity_ids=clients.identity_ids,
    )
    outcome = await run_workflow(runtime, clients)

EVIDENCE = ROOT / "evidence" / outcome.run_id
EVIDENCE.mkdir(parents=True, exist_ok=True)
(EVIDENCE / "agent-outcome.json").write_text(outcome.model_dump_json(indent=2))

# Delta output schemas use strings for stable serialization; cast financial measures to decimal(20,2).
# No local database on OneLake: persist authoritative analysis and trace as append/replace run partitions.
from pyspark.sql.types import StructType, StructField, StringType

def persist(name, records):
    if not records:
        return
    rows = [{"run_id": outcome.run_id, "as_of_date": AS_OF, **r} for r in records]
    fields = sorted({key for row in rows for key in row})
    cooked = [{k: (json.dumps(row[k]) if isinstance(row.get(k), (dict, list)) else
                   str(row[k]) if row.get(k) is not None else None) for k in fields} for row in rows]
    schema = StructType([StructField(k, StringType(), True) for k in fields])
    (spark.createDataFrame(cooked, schema).write.format("delta").mode("overwrite")
        .partitionBy("run_id").option("replaceWhere", f"run_id = '{outcome.run_id}'").saveAsTable(name))

persist("agent_outcomes", [outcome.model_dump(mode="json")])
persist("agent_assessments", runtime.trace)
if outcome.status == "awaiting_human_approval":
    run = runtime.final_run()
    (EVIDENCE / "run.json").write_text(json.dumps(run, indent=2))
    for table in ("positions", "recommendations", "exceptions", "crosswalk", "alerts"):
        persist(table, run[table])
    persist("reconciliation", run["reconciliation"]["rows"])
    persist("run_summary", [dict(**run["summary"], alert_threshold=run["policy"]["alert_value"])])
    persist("agent_handoffs", run["handoffs"])
    persist("weekly_reports", [{"role": k, "report": v} for k, v in run["weekly_reports"].items()])
print(json.dumps({"status": outcome.status, "run_id": outcome.run_id, "evidence": str(EVIDENCE)}))
